# 🧠 Multi-Agent Computer Architecture Teaching Assistant

This notebook runs the **multi-agent RAG system** using LangChain + LangGraph.

**Agents:**
- 📚 **RAG Agent** — Textbook search & Q&A
- 🧮 **Math Agent** — Calculations, CPI, Amdahl's Law
- 📖 **Knowledge Agent** — Definitions, summaries, comparisons
- 💻 **Code Agent** — Assembly code tracing & generation

**Tools:** Textbook Search, Calculator, Glossary, Chapter Summarizer, Concept Comparison, Assembly Tracer, Web Search

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Clone or navigate to project
PROJECT_DIR = '/content/drive/My Drive/Colab_RAG_Project'
if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    print(f'✅ Working directory: {os.getcwd()}')
else:
    print(f'⚠️ Project directory not found at {PROJECT_DIR}')
    print('Please update PROJECT_DIR to point to your project folder.')

In [ ]:
# Cell 2: Install dependencies
!pip install -q langchain>=0.2.0 langchain-community>=0.2.0 langchain-huggingface>=0.0.3
!pip install -q langgraph>=0.1.0
!pip install -q transformers==4.43.3 accelerate sentence-transformers bitsandbytes
!pip install -q faiss-cpu --no-deps
!pip install -q gradio>=4.0.0 duckduckgo-search>=5.0.0 numexpr>=2.8.0
print('✅ All dependencies installed.')

In [ ]:
# Cell 3: Initialize the multi-agent system
from agents.config import Config
from agents.graph import build_graph, run_query
from agents.memory import ConversationMemory

# Configure for Colab
config = Config(
    chapters_db_path='/content/drive/My Drive/Colab_RAG_Project/chapters_db',
    model_name='microsoft/Phi-3.5-mini-instruct',
)
print(config.summary())

# Build the agent graph (loads LLM + initializes all tools)
graph = build_graph(config)
memory = ConversationMemory(max_turns=config.max_history)

print('\n\n✅ Multi-Agent System Ready!')

In [ ]:
# Cell 4: Test the agents

test_queries = [
    'What is the difference between RISC and CISC?',    # -> RAG Agent
    'Calculate 2**10',                                    # -> Math Agent
    'Define pipeline hazard',                             # -> Knowledge Agent
    'Write MIPS code to add two numbers',                 # -> Code Agent
]

for query in test_queries:
    print(f'\n{"="*60}')
    print(f'📝 Query: {query}')
    print(f'{"="*60}')
    
    result = run_query(graph, query, memory)
    
    print(f'\n🤖 Agent: {result["agent_used"]}')
    print(f'\n{result["response"]}')
    
    if result.get('tool_calls_log'):
        print(f'\n🔧 Tools: {", ".join(result["tool_calls_log"])}')
    print()

In [ ]:
# Cell 5: Launch Gradio Chat UI (Interactive)
from app import create_ui, initialize as app_init

# Re-use the already loaded graph and memory
import app as app_module
app_module.graph = graph
app_module.memory = memory
app_module.config = config

demo = create_ui()
demo.launch(share=True, debug=True)  # share=True creates public URL

In [ ]:
# Cell 6: Alternative — Simple Chat Loop (no Gradio)
print('🤖 Agent is ready! Type "exit" to stop.\n')

while True:
    user_input = input('\nYou: ')
    
    if user_input.lower() in ['exit', 'quit', 'stop']:
        print('Goodbye!')
        break
    
    try:
        result = run_query(graph, user_input, memory)
        
        agent_names = {
            'rag_agent': '📚 Textbook',
            'math_agent': '🧮 Math',
            'knowledge_agent': '📖 Knowledge',
            'code_agent': '💻 Code',
            'direct': '💬 Chat',
        }
        agent_label = agent_names.get(result['agent_used'], result['agent_used'])
        
        print(f'\n[{agent_label} Agent]')
        print(f'Assistant: {result["response"]}')
        
        if result.get('tool_calls_log'):
            print(f'\n🔧 Tools: {", ".join(result["tool_calls_log"])}')
        print('-' * 50)
    except Exception as e:
        print(f'❌ Error: {e}')